# 第 1 课｜从招聘数据库到可分析的数据集

**90 分钟：问题定义 15′ → Python/Pandas 25′ → SQLite/JSON 35′ → 质量检查 15′**

本课结束，你能解释“一行是什么”、读取 SQLite、拆开 JSON，并交付可追溯的职位明细表。

业务任务：描述这批官网职位样本的公司、地点与招聘类型结构，为技能分析做准备。
数据分析是“提出问题 → 定义指标 → 整理数据 → 比较与解释 → 支持决策”，不等于画图。

本项目采集工作已完成；你的任务是分析。职位条目不是招聘人数；历史保留、来源覆盖差异、
队列同步删除都会影响样本。我们分析的是固定快照中的待同步记录，不能称为当前全网职位总量。

**运行顺序**：四本 Notebook 按 01→04 执行，每本均从顶部运行。教师版已保留真实执行输出。

In [1]:
from pathlib import Path
import sys, json, re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# 支持从项目根目录、课程目录或 notebooks 目录启动。
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'course_utils.py').exists()), None)
if ROOT is None:
    ROOT = next((p / 'data_analysis202612' for p in candidates
                 if (p / 'data_analysis202612' / 'course_utils.py').exists()), None)
assert ROOT is not None, '请从项目根目录或课程目录启动 Jupyter'
sys.path.insert(0, str(ROOT))
from course_utils import *
pd.set_option('display.max_columns', 14)
pd.set_option('display.max_colwidth', 65)
pd.set_option('display.max_rows', 20)
print('课程目录：', ROOT)
print('Python：', sys.version.split()[0])
print('解释器：', sys.executable)

课程目录： /Users/luosen/WorkBuddy/JobsCrawler/data_analysis202612
Python： 3.10.16
解释器： /Users/luosen/WorkBuddy/JobsCrawler/data_analysis202612/.venv/bin/python


## 1. 必要的 Python：字典、列表、函数、DataFrame

字典像一条职位，列表像多条职位；函数把重复规则封装起来。
DataFrame 是有行列标签的表格，`loc` 筛选、`groupby` 分组、`agg` 聚合。
下面用小数据先练习，再处理真实数据。缺失值不是零。

In [2]:
examples = [
    {'title': '数据分析师', 'city': '北京', 'years': 2},
    {'title': '算法实习生', 'city': '上海', 'years': None},
    {'title': '商业分析师', 'city': '北京', 'years': 3},
]
def title_contains(job, keyword):
    return keyword in job['title']

print([x['title'] for x in examples if title_contains(x, '分析')])
demo = pd.DataFrame(examples)
display(demo)
display(demo.groupby('city', as_index=False).agg(岗位数=('title', 'size'), 已知年限均值=('years', 'mean')))
display(demo.loc[demo['years'].ge(2), ['title', 'city']])

['数据分析师', '商业分析师']


,title,city,years
0,数据分析师,北京,2.0
1,算法实习生,上海,NaN
2,商业分析师,北京,3.0


,city,岗位数,已知年限均值
0,上海,1,NaN
1,北京,2,2.5


,title,city
0,数据分析师,北京
2,商业分析师,北京


**练习 1（3 分钟）**：统计示例中经验年限缺失的数量；把筛选条件改成“岗位名称包含分析”。
预测 `size` 与 `count` 在缺失值存在时会不会相同，再运行验证。

In [3]:
# 参考解答：课堂可先折叠本单元，学生在上方新增代码单元练习。
print('年限缺失数：', demo['years'].isna().sum())
display(demo.loc[demo['title'].str.contains('分析', na=False)])
display(demo.groupby('city')['years'].agg(['size', 'count']))
assert demo['years'].isna().sum() == 1

年限缺失数： 1


,title,city,years
0,数据分析师,北京,2.0
2,商业分析师,北京,3.0


,size,count
city,,
上海,1,0
北京,2,2


## 2. 建立一致、只读的原始快照

直接复制 `.db` 可能遗漏 WAL 中已提交的记录。这里用 SQLite backup API 读取原库，
生成课程快照，并记录 SHA-256、创建时间、表行数和完整性校验。已有快照不会自动覆盖。
原库不执行写入，后续课程只读取这份固定快照。复制过程由 `course_utils.py` 封装。

In [4]:
snapshot, manifest = snapshot_database()
display(pd.Series(manifest, name='数据接收台账'))
import sqlite3
with sqlite3.connect(snapshot.as_uri() + '?mode=ro', uri=True) as conn:
    schema = pd.read_sql_query("SELECT name, sql FROM sqlite_master WHERE type='table'", conn)
    raw = pd.read_sql_query('SELECT * FROM pending_jobs ORDER BY seq', conn)
display(schema)
print('快照职位数：', len(raw), '；原始列数：', len(raw.columns))
assert len(raw) == manifest['table_rows']['pending_jobs']

source                   /Users/luosen/WorkBuddy/JobsCrawler/state/pending_queue.db
snapshot           /Users/luosen/WorkBuddy/JobsCrawler/data_analysis202612/data/...
created_at                                         2026-10-28T17:07:40.124351+08:00
method                    SQLite backup API; source mode=ro; includes committed WAL
sha256             29efa641bd5db92362ae97b75e226b994e5148a2bfe8cb442e397c0860f413ac
size_bytes                                                                335814656
schema_version                                                                    5
integrity_check                                                                  ok
table_rows         {'pending_jobs': 40661, 'sqlite_sequence': 1, 'job_fingerprin...
Name: 数据接收台账, dtype: object

,name,sql
0,pending_jobs,CREATE TABLE pending_jobs (\n seq INTEGER PRIM...
1,sqlite_sequence,"CREATE TABLE sqlite_sequence(name,seq)"
2,job_fingerprints,"CREATE TABLE job_fingerprints (\n source TEXT,..."
3,job_organizations,CREATE TABLE job_organizations (\n source TEXT...
4,api_cache,"CREATE TABLE api_cache(cache_key TEXT PRIMARY KEY,version TEX..."
5,recruitment_batches,"CREATE TABLE recruitment_batches(source TEXT,batch_id TEXT,re..."


快照职位数： 40661 ；原始列数： 9


## 3. 展开 payload，并防止同名字段相互覆盖

`json.loads` 把 JSON 文本变为字典；`json_normalize(max_level=0)` 展开顶层字段，
保留数组/对象。若直接与队列表横向合并，`source`、`job_id` 会重名，所以先加 `payload.` 前缀。
坏 JSON 单独隔离，身份键不一致则停止处理。`portal` 是渠道标识，不强行等同于 `recruitment_type`。

In [5]:
records, valid_indices, rejected = [], [], []
for idx, value in raw['payload'].items():
    try:
        parsed = json.loads(value)
        if not isinstance(parsed, dict):
            raise ValueError('payload 不是对象')
        records.append(parsed)
        valid_indices.append(idx)
    except (ValueError, TypeError) as exc:
        rejected.append({'seq': int(raw.loc[idx, 'seq']), 'reason': str(exc)})

payload = pd.json_normalize(records, max_level=0)
payload.index = valid_indices
flat = raw.loc[valid_indices].drop(columns='payload').join(payload.add_prefix('payload.'))
identity_mismatches = {}
for col in ['source', 'job_id', 'batch_id', 'source_site']:
    a = flat[col].fillna('').astype(str)
    b = flat['payload.' + col].fillna('').astype(str)
    identity_mismatches[col] = int(a.ne(b).sum())
assert not any(identity_mismatches.values()), identity_mismatches
assert not flat.duplicated(KEY).any(), '完整职位身份键重复'

# 保留队列身份列；raw_data 原文仍在快照，不复制到轻量分析表。
business_cols = [c for c in payload.columns if c not in ['source', 'job_id', 'batch_id', 'source_site', 'raw_data']]
jobs = flat[list(raw.columns.drop('payload'))].copy()
for col in business_cols:
    jobs[col] = flat['payload.' + col]
jobs = jobs.reset_index(drop=True)
jobs['job_key'] = jobs['seq'].astype(str)  # 仅在本固定快照内使用的便捷行键
display(jobs[['job_key', 'source', 'portal', 'title', 'city_name', 'recruitment_type']].head())
print('payload 顶层字段数：', len(payload.columns), '；隔离记录数：', len(rejected))

,job_key,source,portal,title,city_name,recruitment_type
0,1,bytedance,society,用户产品经理（造世界方向） - 抖音社交与互动,北京,society
1,2,bytedance,society,游戏资深3D场景模型设计师 - 心光工作室,杭州,society
2,3,bytedance,society,游戏音乐设计师（PVE射击项目） - 研发支持中心,上海,society
3,4,bytedance,society,芯片综合/STA工程师（存算与AI） - 芯片研发,北京,society
4,5,bytedance,society,东南亚跨境IP及主题活动运营-TikTok Shop,上海,society


payload 顶层字段数： 48 ；隔离记录数： 0


## 4. 保留原文，新增清洗列

清洗不是把空值随意补齐。这里去掉 HTML、规范空白，对标题缺失做标记；
学历、薪资保留缺失，不推测。原始城市字段保留，清洗后的地点标签另存。

真实差异：字节城市列表含字典，百度可能一个元素含多个城市，讯飞使用“安徽省·合肥市”，
京东有省级地点，携程有英文城市。城市映射是可审计规则，不是完美地理编码。

In [6]:
jobs['company_name'] = jobs['source'].map(COMPANIES).fillna(jobs['source'])
for col in ['title', 'description', 'requirement']:
    jobs[col + '_clean'] = jobs[col].map(clean_text)
jobs['text_missing'] = jobs['description_clean'].eq('') & jobs['requirement_clean'].eq('')
jobs['title_missing'] = jobs['title_clean'].eq('')
jobs['locations'] = [normalize_locations(a, b) for a, b in zip(jobs['city_list'], jobs['city_name'])]
jobs['cities'] = jobs['locations'].map(lambda xs: [x for x in xs if location_level(x) == '城市候选'])
jobs['city_count'] = jobs['cities'].map(len)
jobs['publish_datetime'] = pd.to_datetime(jobs['publish_time'], errors='coerce', utc=True, format='mixed')

location_audit = jobs[['job_key', 'source', 'locations']].explode('locations').dropna(subset=['locations'])
location_audit['level'] = location_audit['locations'].map(location_level)
location_counts = location_audit.groupby(['locations', 'level'], as_index=False).agg(关联数=('job_key', 'nunique'))
display(location_counts.sort_values('关联数', ascending=False).head(20))
print('人工核验候选：')
display(location_counts.loc[location_counts['level'].ne('城市候选')].sort_values('关联数', ascending=False).head(20))

,locations,level,关联数
60,北京,城市候选,19363
16,上海,城市候选,13076
196,杭州,城市候选,5892
258,深圳,城市候选,4952
150,广州,城市候选,1510
170,成都,城市候选,886
73,合肥,城市候选,621
322,西安,城市候选,311
205,武汉,城市候选,298
144,广东省,省级或地区,213


人工核验候选：


,locations,level,关联数
144,广东省,省级或地区,213
126,山东省,省级或地区,49
210,江苏省,省级或地区,48
382,香港特别行政区,区县待核验,45
335,辽宁省,省级或地区,40
263,湖北省,省级或地区,39
91,四川省,省级或地区,39
218,江西省,省级或地区,35
246,浙江省,省级或地区,32
229,河南省,省级或地区,30


## 5. 数据质量画像与导出

分来源检查字段覆盖率，避免把“解析器没有提供学历”误解成“公司不看学历”。
主表保持一行一条职位；多城市分析下一课再展开。
Parquet 保存列表类型和空值，比把嵌套对象直接塞进 CSV 更适合课间传递。

In [7]:
quality = jobs.groupby('company_name').agg(
    岗位数=('job_key', 'size'), 标题缺失率=('title_missing', 'mean'),
    文本全缺失率=('text_missing', 'mean'), 有城市候选率=('city_count', lambda s: s.gt(0).mean()),
    学历字段有值率=('required_degree', lambda s: s.fillna('').astype(str).str.strip().ne('').mean()),
    发布日期可解析率=('publish_datetime', lambda s: s.notna().mean()),
)
display(quality.round(3))
cleaning_log = {
    'raw_rows': len(raw), 'clean_rows': len(jobs), 'rejected_json_rows': len(rejected),
    'duplicate_identity_rows': int(jobs.duplicated(KEY).sum()),
    'identity_mismatches': identity_mismatches,
    'title_missing_rows': int(jobs['title_missing'].sum()),
    'text_missing_rows': int(jobs['text_missing'].sum()),
    'no_city_candidate_rows': int(jobs['city_count'].eq(0).sum()),
    'missing_publish_time_rows': int(jobs['publish_time'].fillna('').astype(str).str.strip().eq('').sum()),
    'unparseable_nonempty_publish_time_rows': int((
        jobs['publish_time'].fillna('').astype(str).str.strip().ne('') & jobs['publish_datetime'].isna()).sum()),
    'unparsed_or_missing_publish_time_rows': int(jobs['publish_datetime'].isna().sum()),
    'location_associations_by_level': location_audit['level'].value_counts().to_dict(),
    'actions': ['隔离坏JSON', '验证复合身份键', '原文保留并新增清洗列', '城市别名映射与地点层级标记',
                '保留多城市', '薪资学历不填0', '复杂字段JSON序列化以稳定跨课传递'],
}
write_json(REPORTS / 'cleaning_log.json', cleaning_log)
write_json(REPORTS / 'rejected_json.json', rejected)
csv_export(quality.reset_index(), '01_source_quality')
csv_export(location_counts, '01_location_audit')

# 复杂列统一存 JSON 字符串，避免不同来源结构冲突；下课只解码所需列表。
export = jobs.copy()
for col in export.columns:
    if export[col].map(lambda x: isinstance(x, (list, dict))).any():
        export[col] = export[col].map(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else None)
export.to_parquet(DATA / 'processed' / 'jobs_clean.parquet', index=False)
jobs[['job_key', 'source', 'title_clean', 'city_name', 'recruitment_type', 'job_url']].to_csv(
    DATA / 'processed' / 'jobs_preview.csv', index=False, encoding='utf-8-sig')
assert len(jobs) + len(rejected) == len(raw)
assert jobs['job_key'].is_unique
display(pd.Series(cleaning_log))

,岗位数,标题缺失率,文本全缺失率,有城市候选率,学历字段有值率,发布日期可解析率
company_name,,,,,,
OPPO,273,0.0,0.000,0.996,0.546,1.000
京东,2025,0.0,0.000,0.623,0.000,1.000
哔哩哔哩,872,0.0,0.001,1.000,0.000,1.000
字节跳动,17053,0.0,0.000,1.000,0.000,1.000
小红书,1299,0.0,0.000,0.999,0.000,1.000
快手,2563,0.0,0.000,1.000,0.000,0.101
拼多多,767,0.0,0.000,0.986,0.000,1.000
携程,534,0.0,0.000,0.957,0.000,1.000
滴滴,1808,0.0,0.000,0.928,0.320,1.000


raw_rows                                                                                             40661
clean_rows                                                                                           40661
rejected_json_rows                                                                                       0
duplicate_identity_rows                                                                                  0
identity_mismatches                            {'source': 0, 'job_id': 0, 'batch_id': 0, 'source_site': 0}
title_missing_rows                                                                                       0
text_missing_rows                                                                                        3
no_city_candidate_rows                                                                                 972
missing_publish_time_rows                                                                             6470
unparseable_nonempty_publish_time_row

## 课堂练习与验收

1. 查看 `raw.iloc[0]['payload']` 的类型，再对比 `json.loads(...)` 的类型。
2. 为什么只用 `job_id` 去重不可靠？请给出两家公司可能使用同一 ID 的例子。
3. 找出城市数最多的 5 条职位，检查原始城市字段和清洗后列表。
4. 从质量表选择一个缺失率高的字段，解释为什么它不适合直接跨公司比较。

**参考解答思路**：原始 payload 是字符串、解析后是字典；职位身份包含来源、渠道、批次与站点；
多城市不能压成一个随意选择的城市；缺失可能来自接口或解析覆盖差异。

**验收**：原始行数=清洗行数+隔离行数，复合键唯一，原文仍可回溯，导出成功。
下一课读取 `data/processed/jobs_clean.parquet`。

In [8]:
display(jobs.sort_values('city_count', ascending=False)[['source', 'title_clean', 'city_name', 'cities']].head(5))
print('第1课完成：', DATA / 'processed' / 'jobs_clean.parquet')

,source,title_clean,city_name,cities
28908,meituan,进场零售-商场CKA,广州市、武汉市、安庆市、亳州市、长沙市、常州市、池州市、滁州市、鄂州市、恩施土家族苗族自治州、抚州市、阜阳市、赣州市、贵阳...,"[广州, 武汉, 安庆, 亳州, 长沙, 常州, 池州, 滁州, 鄂州, 抚州, 阜阳, 赣州, 贵阳, 杭州, 合肥, ..."
29277,meituan,营销顾问/商务拓展,厦门市、福州市、杭州市、无锡市、宁波市、北京市、温州市、苏州市、天津市、惠州市、石家庄市、上海市、成都市、长春市、珠海市、...,"[厦门, 福州, 杭州, 无锡, 宁波, 北京, 温州, 苏州, 天津, 惠州, 石家庄, 上海, 成都, 长春, 珠海,..."
28526,meituan,医药健康-城市直营中心-SMB,北京市、呼和浩特市、沈阳市、大连市、哈尔滨市、南京市、扬州市、杭州市、宁波市、温州市、嘉兴市、绍兴市、金华市、芜湖市、福州...,"[北京, 呼和浩特, 沈阳, 大连, 哈尔滨, 南京, 扬州, 杭州, 宁波, 温州, 嘉兴, 绍兴, 金华, 芜湖, 福..."
28959,meituan,火石-酒店项目-大客户销售,北京市、天津市、沈阳市、大连市、上海市、南京市、南通市、常州市、无锡市、苏州市、杭州市、宁波市、嘉兴市、合肥市、福州市、厦...,"[北京, 天津, 沈阳, 大连, 上海, 南京, 南通, 常州, 无锡, 苏州, 杭州, 宁波, 嘉兴, 合肥, 福州, ..."
29372,meituan,门店管理储备岗,杭州市、宁波市、无锡市、北京市、苏州市、天津市、上海市、石家庄市、成都市、珠海市、济南市、青岛市、广州市、深圳市、武汉市、...,"[杭州, 宁波, 无锡, 北京, 苏州, 天津, 上海, 石家庄, 成都, 珠海, 济南, 青岛, 广州, 深圳, 武汉,..."


第1课完成： /Users/luosen/WorkBuddy/JobsCrawler/data_analysis202612/data/processed/jobs_clean.parquet
